# 05 — Training: ResNet-50 (Local — RTX 5060 Ti)

Entrenamiento de **ResNet-50** preentrenado en ImageNet para clasificación morfológica de galaxias (6 clases).  
Versión local optimizada para **NVIDIA RTX 5060 Ti** (Blackwell, sm_120).

| Hiperparámetro | Valor |
|---|---|
| Batch size | 64 (8 GB VRAM) — subir a 96/128 si tienes 16 GB |
| LR backbone | 1e-4 |
| LR head | 1e-3 |
| Optimizer | AdamW (wd=1e-4) |
| Scheduler | CosineAnnealingLR |
| Epochs | 30 (+ early stopping, paciencia=5) |
| AMP | ✅ float16 |
| torch.compile | ✅ |
| Early stopping | ✅ paciencia=5 epochs sin mejora en val F1 |

> **Lección de EfficientNet-B3:** el mejor modelo se alcanzó en epoch 16 pero el entrenamiento siguió 14 epochs más sin mejorar.  
> Early stopping evita ese desperdicio y reduce el riesgo de sobreajuste.

> **Requisito de CUDA:** RTX 5060 Ti requiere **CUDA 12.8+** y **PyTorch ≥ 2.7**. Ejecuta primero la celda de instalación.

## Sección 0 — Instalación de dependencias

Ejecuta esta celda **una sola vez**. Reinicia el kernel después si es la primera instalación.

In [ ]:
import subprocess, sys

# PyTorch con CUDA 12.8 (necesario para RTX 5060 Ti / Blackwell sm_120)
subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade',
    'torch', 'torchvision',
    '--index-url', 'https://download.pytorch.org/whl/cu128',
    '--quiet',
], check=True)

subprocess.run([
    sys.executable, '-m', 'pip', 'install', '--upgrade', '--quiet',
    'pandas', 'numpy', 'matplotlib', 'seaborn',
    'Pillow', 'scikit-learn', 'tqdm',
], check=True)

print('Instalación completada. Reinicia el kernel si es la primera vez.')

## Sección 1 — Imports

In [1]:
import gc
import os
import sys
import time
import pathlib
import warnings
from datetime import datetime

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import torch
import torch.nn as nn
import torch.optim as optim
from torch.cuda.amp import autocast, GradScaler
from torch.utils.data import DataLoader, Dataset
from torchvision import transforms
from torchvision.models import resnet50, ResNet50_Weights
from sklearn.metrics import f1_score, classification_report, confusion_matrix
from PIL import Image
from tqdm.auto import tqdm

warnings.filterwarnings('ignore')

print(f'PyTorch  : {torch.__version__}')
print(f'CUDA     : {torch.cuda.is_available()}')
if torch.cuda.is_available():
    print(f'GPU      : {torch.cuda.get_device_name(0)}')
    print(f'VRAM     : {torch.cuda.get_device_properties(0).total_memory / 1e9:.1f} GB')
    cap = torch.cuda.get_device_capability(0)
    print(f'Compute  : sm_{cap[0]}{cap[1]}')

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(f'Device   : {device}')

PyTorch  : 2.11.0+cu128
CUDA     : True
GPU      : NVIDIA GeForce RTX 5060 Ti
VRAM     : 17.1 GB
Compute  : sm_120
Device   : cuda


c:\Users\HOME\Desktop\IA\venv\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Sección 2 — Configuración

In [2]:
# Paths locales
_LOCAL     = pathlib.Path('../data')
IMAGES_DIR = _LOCAL / 'images_gz2' / 'images'
SPLITS_DIR = _LOCAL / 'splits'
CKPT_DIR   = pathlib.Path('../models/checkpoints/resnet50')
LOG_DIR    = pathlib.Path('../logs')

CKPT_DIR.mkdir(parents=True, exist_ok=True)
LOG_DIR.mkdir(parents=True, exist_ok=True)

assert IMAGES_DIR.exists(), f'No se encontró IMAGES_DIR: {IMAGES_DIR.resolve()}'
assert (SPLITS_DIR / 'train.csv').exists(), f'No se encontró train.csv en {SPLITS_DIR.resolve()}'

# Model
MODEL_NAME   = 'resnet50'
NUM_CLASSES  = 6
CLASS_ORDER  = ['Elliptical', 'Lenticular', 'Spiral', 'Barred_Spiral', 'Edge_on', 'Irregular']
CLASS_TO_IDX = {c: i for i, c in enumerate(CLASS_ORDER)}
IDX_TO_CLASS = {i: c for c, i in CLASS_TO_IDX.items()}

# Training
EPOCHS          = 30
EARLY_STOP_PAT  = 5     # epochs sin mejora en val F1 antes de detener
BATCH_SIZE      = 128    # seguro para 8 GB VRAM con AMP; subir a 96/128 con 16 GB

# En Windows + Jupyter, num_workers > 0 causa deadlock (workers reimportan el kernel).
NUM_WORKERS     = 0 if os.name == 'nt' else 4

LR_BACKBONE  = 1e-4
LR_HEAD      = 1e-3
WEIGHT_DECAY = 1e-4
USE_AMP      = device.type == 'cuda'
USE_COMPILE  = False

# ResNet-50 fue entrenado con 224×224 (resolución nativa).
# Nuestras imágenes son 424×424 → CenterCrop(280) → Resize(224): margen 1.25×.
IMAGE_SIZE    = 224
CROP_SIZE     = 280
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD  = [0.229, 0.224, 0.225]
RANDOM_SEED   = 42

torch.manual_seed(RANDOM_SEED)
np.random.seed(RANDOM_SEED)

print(f'OS             : {"Windows" if os.name == "nt" else "Linux/macOS"}')
print(f'IMAGES_DIR     : {IMAGES_DIR.resolve()}')
print(f'SPLITS_DIR     : {SPLITS_DIR.resolve()}')
print(f'CKPT_DIR       : {CKPT_DIR.resolve()}')
print(f'MODEL          : {MODEL_NAME}')
print(f'EPOCHS         : {EPOCHS}  (early stop paciencia={EARLY_STOP_PAT})')
print(f'BATCH_SIZE     : {BATCH_SIZE}')
print(f'NUM_WORKERS    : {NUM_WORKERS}')
print(f'LR backbone    : {LR_BACKBONE}')
print(f'LR head        : {LR_HEAD}')
print(f'USE_AMP        : {USE_AMP}')
print(f'USE_COMPILE    : {USE_COMPILE}')

OS             : Windows
IMAGES_DIR     : C:\Users\HOME\Desktop\IA\data\images_gz2\images
SPLITS_DIR     : C:\Users\HOME\Desktop\IA\data\splits
CKPT_DIR       : C:\Users\HOME\Desktop\IA\models\checkpoints\resnet50
MODEL          : resnet50
EPOCHS         : 30  (early stop paciencia=5)
BATCH_SIZE     : 128
NUM_WORKERS    : 0
LR backbone    : 0.0001
LR head        : 0.001
USE_AMP        : True
USE_COMPILE    : False


## Sección 3 — Pipeline de datos

In [3]:
train_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.RandomHorizontalFlip(p=0.5),
    transforms.RandomVerticalFlip(p=0.5),
    transforms.RandomRotation(degrees=180),
    transforms.ColorJitter(brightness=0.15, contrast=0.15),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])

eval_transforms = transforms.Compose([
    transforms.CenterCrop(CROP_SIZE),
    transforms.Resize(IMAGE_SIZE),
    transforms.ToTensor(),
    transforms.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
])


class GalaxyDataset(Dataset):
    """Almacena numpy arrays para evitar copy-on-read en workers forked."""
    def __init__(self, csv_path, images_dir, transform, class_to_idx):
        df = pd.read_csv(
            csv_path,
            usecols=['img_filename', 'morph_label'],
            dtype={'img_filename': 'str', 'morph_label': 'str'},
        )
        self.filenames = df['img_filename'].to_numpy()
        self.labels    = np.array(
            [class_to_idx[lbl] for lbl in df['morph_label']], dtype=np.int64
        )
        del df
        gc.collect()
        self.images_dir = pathlib.Path(images_dir)
        self.transform  = transform

    def __len__(self):
        return len(self.filenames)

    def __getitem__(self, idx):
        image = Image.open(self.images_dir / self.filenames[idx]).convert('RGB')
        if self.transform:
            image = self.transform(image)
        return image, int(self.labels[idx])


g = torch.Generator().manual_seed(RANDOM_SEED)

train_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'train.csv', IMAGES_DIR, train_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'), generator=g,
)
val_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'val.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)

print(f'Train batches : {len(train_loader):,}  ({len(train_loader.dataset):,} imgs)')
print(f'Val   batches : {len(val_loader):,}  ({len(val_loader.dataset):,} imgs)')

Train batches : 608  (77,789 imgs)
Val   batches : 131  (16,670 imgs)


In [4]:
# Class weights para CrossEntropyLoss
_df_w = pd.read_csv(
    SPLITS_DIR / 'train.csv',
    usecols=['morph_label'],
    dtype={'morph_label': 'str'},
)
label_counts  = np.bincount(_df_w['morph_label'].map(CLASS_TO_IDX).values, minlength=NUM_CLASSES)
weights       = len(_df_w) / (NUM_CLASSES * label_counts)
class_weights = torch.tensor(weights, dtype=torch.float32).to(device)

print('Class weights:')
for cls, w, n in zip(CLASS_ORDER, weights, label_counts):
    print(f'  {cls:<15}  n={n:>6,}   w={w:.4f}')

del _df_w
gc.collect()
print('RAM liberada ✓')

Class weights:
  Elliptical       n=17,500   w=0.7408
  Lenticular       n=11,848   w=1.0943
  Spiral           n=17,500   w=0.7408
  Barred_Spiral    n=17,500   w=0.7408
  Edge_on          n= 9,292   w=1.3953
  Irregular        n= 4,149   w=3.1248
RAM liberada ✓


## Sección 4 — Modelo

In [5]:
# ResNet-50 con pesos ImageNet V2 (mejor calibración que V1)
base_model = resnet50(weights=ResNet50_Weights.IMAGENET1K_V2)

# Reemplazar la capa fully-connected final (2048 → NUM_CLASSES)
in_features = base_model.fc.in_features
base_model.fc = nn.Linear(in_features, NUM_CLASSES)
print(f'FC head: Linear({in_features}, {NUM_CLASSES})')

# Grupos de parámetros ANTES de torch.compile()
backbone_params = [p for n, p in base_model.named_parameters() if not n.startswith('fc')]
head_params     = list(base_model.fc.parameters())
print(f'Backbone params : {sum(p.numel() for p in backbone_params):,}')
print(f'Head params     : {sum(p.numel() for p in head_params):,}')

model = base_model.to(device)

if USE_COMPILE and hasattr(torch, 'compile'):
    model = torch.compile(model)
    print('torch.compile() aplicado ✓')
else:
    print('torch.compile() no disponible o desactivado')

print(f'Total params    : {sum(p.numel() for p in model.parameters()):,}')

FC head: Linear(2048, 6)
Backbone params : 23,508,032
Head params     : 12,294
torch.compile() no disponible o desactivado
Total params    : 23,520,326


## Sección 5 — Infraestructura de entrenamiento

In [6]:
criterion = nn.CrossEntropyLoss(weight=class_weights)

optimizer = optim.AdamW(
    [
        {'params': backbone_params, 'lr': LR_BACKBONE},
        {'params': head_params,     'lr': LR_HEAD},
    ],
    weight_decay=WEIGHT_DECAY,
)

scheduler = optim.lr_scheduler.CosineAnnealingLR(
    optimizer, T_max=EPOCHS, eta_min=1e-6
)

scaler = GradScaler(enabled=USE_AMP)

print('Loss     : CrossEntropyLoss (weighted)')
print('Optim    : AdamW')
print('Schedule : CosineAnnealingLR')
print(f'AMP      : {USE_AMP}')

Loss     : CrossEntropyLoss (weighted)
Optim    : AdamW
Schedule : CosineAnnealingLR
AMP      : True


In [7]:
def _unwrap_model(m: nn.Module) -> nn.Module:
    """Extrae el módulo base de DataParallel o torch.compile()."""
    if isinstance(m, nn.DataParallel):
        m = m.module
    if hasattr(m, '_orig_mod'):  # torch.compile() envuelve en _orig_mod
        m = m._orig_mod
    return m


def save_checkpoint(state: dict, ckpt_dir: pathlib.Path, is_best: bool = False):
    """
    - latest.pth    : sobreescrito en cada epoch
    - best.pth      : solo cuando mejora el val F1
    - epoch_XXX.pth : hito cada 5 epochs
    """
    ckpt_dir.mkdir(parents=True, exist_ok=True)
    torch.save(state, ckpt_dir / 'latest.pth')
    saved = ['latest.pth']
    if is_best:
        torch.save(state, ckpt_dir / 'best.pth')
        saved.append('best.pth')
    if (state['epoch'] + 1) % 5 == 0:
        name = f'epoch_{state["epoch"]+1:03d}.pth'
        torch.save(state, ckpt_dir / name)
        saved.append(name)
    print(f'  [ckpt] saved: {", ".join(saved)}')


def load_checkpoint(
    ckpt_path: pathlib.Path,
    model: nn.Module,
    optimizer: optim.Optimizer,
    scheduler,
    scaler: GradScaler,
):
    """
    Restaura modelo, optimizer, scheduler, scaler e historial.
    Compatible con checkpoints de Kaggle (DataParallel) y local.
    Devuelve (last_epoch, best_val_f1, epochs_no_improve, history).
    """
    ckpt = torch.load(ckpt_path, map_location='cpu')
    _unwrap_model(model).load_state_dict(ckpt['model_state_dict'])
    optimizer.load_state_dict(ckpt['optimizer_state_dict'])
    scheduler.load_state_dict(ckpt['scheduler_state_dict'])
    if 'scaler_state_dict' in ckpt:
        scaler.load_state_dict(ckpt['scaler_state_dict'])
    epochs_no_improve = ckpt.get('epochs_no_improve', 0)
    return ckpt['epoch'], ckpt['best_val_f1'], epochs_no_improve, ckpt['history']


print('Funciones de checkpoint definidas ✓')

Funciones de checkpoint definidas ✓


## Sección 6 — Reanudar desde checkpoint

Si existe `latest.pth` en `CKPT_DIR`, el entrenamiento continúa desde el siguiente epoch (incluyendo el contador de early stopping).  
Para empezar desde cero, borra o renombra `latest.pth`.

In [8]:
LATEST_CKPT = CKPT_DIR / 'latest.pth'

start_epoch       = 0
best_val_f1       = 0.0
epochs_no_improve = 0    # contador para early stopping
history           = []

if LATEST_CKPT.exists():
    print(f'Checkpoint encontrado: {LATEST_CKPT}')
    last_epoch, best_val_f1, epochs_no_improve, history = load_checkpoint(
        LATEST_CKPT, model, optimizer, scheduler, scaler
    )
    start_epoch = last_epoch + 1
    print(f'Reanudando desde epoch {start_epoch + 1}/{EPOCHS}')
    print(f'Mejor val F1         : {best_val_f1:.4f}')
    print(f'Epochs sin mejora    : {epochs_no_improve}/{EARLY_STOP_PAT}')
    print(f'Epochs en historial  : {len(history)}')
else:
    print('Sin checkpoint — entrenamiento desde cero')

print(f'Epochs por entrenar  : {EPOCHS - start_epoch}')

Sin checkpoint — entrenamiento desde cero
Epochs por entrenar  : 30


## Sección 7 — Funciones de entrenamiento y validación

In [9]:
def train_one_epoch(model, loader, optimizer, criterion, scaler, device, use_amp):
    model.train()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='  Train', unit='batch', leave=False, dynamic_ncols=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)

        optimizer.zero_grad()
        with autocast(enabled=use_amp):
            logits = model(imgs)
            loss   = criterion(logits, labels)

        scaler.scale(loss).backward()
        scaler.step(optimizer)
        scaler.update()

        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(logits.argmax(dim=1).detach().cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = running_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1


@torch.no_grad()
def validate(model, loader, criterion, device, use_amp):
    model.eval()
    running_loss = 0.0
    all_preds, all_labels = [], []

    pbar = tqdm(loader, desc='  Val  ', unit='batch', leave=False, dynamic_ncols=True)
    for imgs, labels in pbar:
        imgs   = imgs.to(device, non_blocking=True)
        labels = labels.to(device, non_blocking=True)
        with autocast(enabled=use_amp):
            logits = model(imgs)
            loss   = criterion(logits, labels)
        running_loss += loss.item() * imgs.size(0)
        all_preds.extend(logits.argmax(dim=1).cpu().tolist())
        all_labels.extend(labels.cpu().tolist())
        pbar.set_postfix(loss=f'{loss.item():.4f}')

    avg_loss = running_loss / len(loader.dataset)
    f1       = f1_score(all_labels, all_preds, average='macro', zero_division=0)
    return avg_loss, f1, all_preds, all_labels


print('train_one_epoch / validate definidas ✓')

train_one_epoch / validate definidas ✓


## Sección 8 — Loop de entrenamiento

In [10]:
LOG_CSV = LOG_DIR / f'{MODEL_NAME}_log.csv'

print(f'Iniciando entrenamiento: epochs {start_epoch+1} → {EPOCHS}')
print(f'Early stopping: paciencia = {EARLY_STOP_PAT} epochs sin mejora en val F1')
print('=' * 72)

stopped_early = False

for epoch in range(start_epoch, EPOCHS):
    gc.collect()
    if device.type == 'cuda':
        torch.cuda.empty_cache()

    t0 = time.time()
    ts = datetime.now().strftime('%H:%M:%S')
    print(f'\n[{ts}] ── Epoch {epoch+1:02d}/{EPOCHS} ───────────────────────────────')

    train_loss, train_f1 = train_one_epoch(
        model, train_loader, optimizer, criterion, scaler, device, USE_AMP
    )
    val_loss, val_f1, _, _ = validate(
        model, val_loader, criterion, device, USE_AMP
    )
    scheduler.step()

    elapsed = time.time() - t0
    is_best  = val_f1 > best_val_f1

    if is_best:
        best_val_f1       = val_f1
        epochs_no_improve = 0
    else:
        epochs_no_improve += 1

    row = {
        'epoch':       epoch + 1,
        'train_loss':  round(train_loss, 6),
        'train_f1':    round(train_f1,   6),
        'val_loss':    round(val_loss,   6),
        'val_f1':      round(val_f1,     6),
        'lr_backbone': round(optimizer.param_groups[0]['lr'], 8),
        'lr_head':     round(optimizer.param_groups[1]['lr'], 8),
        'elapsed_s':   round(elapsed, 1),
        'is_best':     is_best,
    }
    history.append(row)

    state = {
        'epoch':                epoch,
        'model_state_dict':     _unwrap_model(model).state_dict(),
        'optimizer_state_dict': optimizer.state_dict(),
        'scheduler_state_dict': scheduler.state_dict(),
        'scaler_state_dict':    scaler.state_dict(),
        'best_val_f1':          best_val_f1,
        'epochs_no_improve':    epochs_no_improve,
        'history':              history,
    }
    save_checkpoint(state, CKPT_DIR, is_best=is_best)
    pd.DataFrame(history).to_csv(LOG_CSV, index=False)

    best_tag   = '  ← BEST' if is_best else ''
    early_tag  = f'  [no mejora: {epochs_no_improve}/{EARLY_STOP_PAT}]' if not is_best else ''
    print(
        f'  train  loss={train_loss:.4f}  F1={train_f1:.4f}\n'
        f'  val    loss={val_loss:.4f}  F1={val_f1:.4f}{best_tag}{early_tag}\n'
        f'  time   {elapsed:.0f}s   LR_bb={optimizer.param_groups[0]["lr"]:.2e}',
        flush=True,
    )

    # Early stopping
    if epochs_no_improve >= EARLY_STOP_PAT:
        print(f'\n[Early Stopping] {EARLY_STOP_PAT} epochs sin mejora — deteniendo en epoch {epoch+1}.')
        stopped_early = True
        break

print('\n' + '=' * 72)
if stopped_early:
    print(f'Entrenamiento detenido por early stopping.  Mejor val F1 = {best_val_f1:.4f}')
else:
    print(f'Entrenamiento completo ({EPOCHS} epochs).  Mejor val F1 = {best_val_f1:.4f}')

Iniciando entrenamiento: epochs 1 → 30
Early stopping: paciencia = 5 epochs sin mejora en val F1

[20:58:14] ── Epoch 01/30 ───────────────────────────────


  [ckpt] saved: latest.pth, best.pth
  train  loss=0.9890  F1=0.5817
  val    loss=0.8466  F1=0.6345  ← BEST
  time   763s   LR_bb=9.97e-05

[21:10:58] ── Epoch 02/30 ───────────────────────────────


  [ckpt] saved: latest.pth
  train  loss=0.8352  F1=0.6358
  val    loss=0.8418  F1=0.6277  [no mejora: 1/5]
  time   1219s   LR_bb=9.89e-05

[21:31:18] ── Epoch 03/30 ───────────────────────────────


  [ckpt] saved: latest.pth, best.pth
  train  loss=0.7915  F1=0.6502
  val    loss=0.7678  F1=0.6570  ← BEST
  time   1105s   LR_bb=9.76e-05

[21:49:44] ── Epoch 04/30 ───────────────────────────────


  [ckpt] saved: latest.pth, best.pth
  train  loss=0.7700  F1=0.6593
  val    loss=0.7503  F1=0.6641  ← BEST
  time   1082s   LR_bb=9.57e-05

[22:07:46] ── Epoch 05/30 ───────────────────────────────


...

  [ckpt] saved: latest.pth, epoch_015.pth
  train  loss=0.5883  F1=0.7302
  val    loss=0.7881  F1=0.6812  [no mejora: 1/5]
  time   609s   LR_bb=5.05e-05

[23:58:23] ── Epoch 16/30 ───────────────────────────────


  [ckpt] saved: latest.pth
  train  loss=0.5664  F1=0.7403
  val    loss=0.7884  F1=0.6798  [no mejora: 2/5]
  time   600s   LR_bb=4.53e-05

[00:08:23] ── Epoch 17/30 ───────────────────────────────


  [ckpt] saved: latest.pth
  train  loss=0.5444  F1=0.7503
  val    loss=0.8215  F1=0.6822  [no mejora: 3/5]
  time   506s   LR_bb=4.02e-05

[00:16:49] ── Epoch 18/30 ───────────────────────────────


  [ckpt] saved: latest.pth
  train  loss=0.5232  F1=0.7614
  val    loss=0.8381  F1=0.6836  [no mejora: 4/5]
  time   499s   LR_bb=3.52e-05

[00:25:08] ── Epoch 19/30 ───────────────────────────────


  [ckpt] saved: latest.pth
  train  loss=0.5028  F1=0.7723
  val    loss=0.8367  F1=0.6816  [no mejora: 5/5]
  time   502s   LR_bb=3.04e-05

[Early Stopping] 5 epochs sin mejora — deteniendo en epoch 19.

Entrenamiento detenido por early stopping.  Mejor val F1 = 0.6914


## Sección 9 — Curvas de entrenamiento

In [ ]:
hist_df = pd.DataFrame(history)

fig, axes = plt.subplots(1, 3, figsize=(18, 5))
fig.suptitle(f'{MODEL_NAME} — Training History', fontsize=13, fontweight='bold')

axes[0].plot(hist_df['epoch'], hist_df['train_loss'], label='Train')
axes[0].plot(hist_df['epoch'], hist_df['val_loss'],   label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss')
axes[0].set_title('Cross-Entropy Loss'); axes[0].legend(); axes[0].grid(alpha=0.3)

best_row = hist_df.loc[hist_df['val_f1'].idxmax()]
axes[1].plot(hist_df['epoch'], hist_df['train_f1'], label='Train')
axes[1].plot(hist_df['epoch'], hist_df['val_f1'],   label='Val')
axes[1].axvline(best_row['epoch'], color='red', linestyle='--', alpha=0.5,
                label=f'Best={best_row["val_f1"]:.4f} (ep{int(best_row["epoch"])})')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('F1 macro')
axes[1].set_title('Macro F1 Score'); axes[1].legend(); axes[1].grid(alpha=0.3)

axes[2].semilogy(hist_df['epoch'], hist_df['lr_backbone'], label='Backbone')
axes[2].semilogy(hist_df['epoch'], hist_df['lr_head'],     label='Head')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('Learning Rate (log)')
axes[2].set_title('Learning Rate Schedule'); axes[2].legend(); axes[2].grid(alpha=0.3)

plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_training_curves.png', dpi=150, bbox_inches='tight')
plt.show()
print(f'Figura guardada en {LOG_DIR}/{MODEL_NAME}_training_curves.png')

## Sección 10 — Evaluación final (mejor modelo sobre test)

In [ ]:
# Cargar el mejor checkpoint
best_ckpt = CKPT_DIR / 'best.pth'
if best_ckpt.exists():
    ckpt = torch.load(best_ckpt, map_location='cpu')
    _unwrap_model(model).load_state_dict(ckpt['model_state_dict'])
    print(f'Mejor modelo: epoch {ckpt["epoch"]+1}  val F1={ckpt["best_val_f1"]:.4f}')
else:
    print('best.pth no encontrado — usando el estado actual del modelo')

# test_loader creado aquí para no mantener workers vivos durante el entrenamiento
test_loader = DataLoader(
    GalaxyDataset(SPLITS_DIR / 'test.csv', IMAGES_DIR, eval_transforms, CLASS_TO_IDX),
    batch_size=BATCH_SIZE, shuffle=False, num_workers=NUM_WORKERS,
    pin_memory=(device.type == 'cuda'),
)
print(f'Test  batches : {len(test_loader):,}  ({len(test_loader.dataset):,} imgs)')

_, _, test_preds, test_labels = validate(model, test_loader, criterion, device, USE_AMP)

print('\nClassification Report — Test set:')
print(classification_report(test_labels, test_preds, target_names=CLASS_ORDER, zero_division=0))

In [ ]:
# Confusion matrix
cm      = confusion_matrix(test_labels, test_preds)
cm_norm = cm.astype(float) / cm.sum(axis=1, keepdims=True)

fig, axes = plt.subplots(1, 2, figsize=(16, 6))
fig.suptitle(f'{MODEL_NAME} — Confusion Matrix (test set)', fontsize=12, fontweight='bold')

for ax, data, title, fmt in [
    (axes[0], cm,      'Counts',     'd'),
    (axes[1], cm_norm, 'Normalized', '.2f'),
]:
    sns.heatmap(
        data, annot=True, fmt=fmt, cmap='Blues',
        xticklabels=CLASS_ORDER, yticklabels=CLASS_ORDER,
        ax=ax, vmin=0, vmax=(1 if fmt == '.2f' else None),
    )
    ax.set_title(title)
    ax.set_xlabel('Predicted')
    ax.set_ylabel('True')
    ax.set_xticklabels(CLASS_ORDER, rotation=30, ha='right')
    ax.set_yticklabels(CLASS_ORDER, rotation=0)

plt.tight_layout()
plt.savefig(LOG_DIR / f'{MODEL_NAME}_confusion_matrix.png', dpi=150, bbox_inches='tight')
plt.show()

## Sección 11 — Resumen

**Artefactos generados:**
- `models/checkpoints/resnet50/latest.pth` — último checkpoint
- `models/checkpoints/resnet50/best.pth` — mejor checkpoint por val F1
- `models/checkpoints/resnet50/epoch_XXX.pth` — hitos cada 5 epochs
- `logs/resnet50_log.csv` — historial epoch por epoch
- `logs/resnet50_training_curves.png`
- `logs/resnet50_confusion_matrix.png`

**Diferencias respecto a EfficientNet-B3:**
- Arquitectura residual clásica (skip connections) vs compuesta (compound scaling)
- Cabeza `fc` (una sola capa) vs `classifier` (Dropout + Linear)
- Mismo IMAGE_SIZE (224) pero CROP_SIZE reducido a 280 (margen 1.25×)
- Early stopping activo — el entrenamiento se detiene automáticamente si no mejora

**Para comparar con EfficientNet-B3:**  
Consultar los dos `_log.csv` en `logs/` y los classification reports del test set.

**Siguiente paso → `06_train_convnext_tiny_local.ipynb`**